# **Data Loading and Preparation**
Load MHC (curated) and IEDB data

**Train and Test Strategy**:
1. Train: 70% of MHC data
2. Validation/Development: 15% of MHC data
3. Test: 15% of MHC and IEDB data (removing overlaps with MHC)

In [19]:
# Importing libraries
import os
import pandas as pd
import re
from collections import Counter
from mhcflurry import downloads
from mhcflurry import Class1AffinityPredictor
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
import shutil
from pathlib import Path

# **MHC Data Loading**
     1) Drop duplicates and NULL values
     2) Remove non-human alleles
     3) Split into train, validation and test split

In [20]:
# Download mhcflurry data if missing, then verify
required = ['models_class1_pan', 'models_class1_presentation', 'data_curated']
for name in required:
    if not os.path.exists(downloads.get_path(name, test_exists=False)):
        print(f'Downloading {name}...')
        downloads.download(name)

# Verify
missing = [n for n in required if not os.path.exists(downloads.get_path(n, test_exists=False))]
print('Loaded' if not missing else f'Missing: {missing}')

Loaded


In [21]:
# Load curated training data, displaying data profile
data_path = downloads.get_path("data_curated", "curated_training_data.csv.bz2")
print(f"Loading curated training data from:\n{data_path}\n")

# Read the data
mhc = pd.read_csv(data_path)

print(f"Dataset shape: {mhc.shape[0]:,} rows × {mhc.shape[1]} columns\n")
print("Columns:", list(mhc.columns))

print("First 10 rows:")
display(mhc.head(10))

# Show summary statistics
print("Dataset Summary:")
print(f"Total entries: {len(mhc):,}")
print(f"Unique peptides: {mhc['peptide'].nunique():,}")
print(f"Unique alleles: {mhc['allele'].nunique():,}")
if 'measurement_value' in mhc.columns:
    print(f"Mean binding affinity: {mhc['measurement_value'].mean():.2f} nM")
    print(f"Median binding affinity: {mhc['measurement_value'].median():.2f} nM")

Loading curated training data from:
/Users/pranavipanduga/Library/Application Support/mhcflurry/4/2.2.0/data_curated/curated_training_data.csv.bz2

Dataset shape: 994,948 rows × 8 columns

Columns: ['allele', 'peptide', 'measurement_value', 'measurement_inequality', 'measurement_type', 'measurement_kind', 'measurement_source', 'original_allele']
First 10 rows:


,allele,peptide,measurement_value,measurement_inequality,measurement_type,measurement_kind,measurement_source,original_allele
0,BoLA-1*009:01,AQVHLGHR,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
1,BoLA-1*009:01,ASLSMENVEVVH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
2,BoLA-1*009:01,EEIAHVLHY,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
3,BoLA-1*009:01,FEYEFPINH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
4,BoLA-1*009:01,GVDVDQLLH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
5,BoLA-1*009:01,HQQDSQYYLTQH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
6,BoLA-1*009:01,LQSEVFPNY,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
7,BoLA-1*009:01,PYGFSTGSGERDLLLSH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
8,BoLA-1*009:01,RTFNDVSKRKH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01
9,BoLA-1*009:01,SQMSRNIESKH,100.0,<,qualitative,mass_spec,Ternette - cellular MHC/mass spectrometry,BoLA-1*009:01


Dataset Summary:
Total entries: 994,948
Unique peptides: 552,042
Unique alleles: 323
Mean binding affinity: 4511.46 nM
Median binding affinity: 100.00 nM


## **1) Drop duplicates and NULL values & 2) Remove non-human alleles**

In [22]:
# Cleaning data and dropping duplicates
mhc.dropna(subset=['peptide', 'allele', 'measurement_value', 'original_allele'])  # Only essential columns
mhc = mhc.drop_duplicates()

# Remove non-human data from df_curated (keep only alleles starting with 'HLA-')
non_human_alleles = mhc.loc[~mhc['allele'].str.startswith('HLA-'), 'allele'].unique()
prefixes = [re.match(r'^[^-]+', allele).group(0) if re.match(r'^[^-]+', allele) else allele for allele in non_human_alleles]
prefix_counts = Counter(prefixes)

print("Non-human allele prefixes and their counts:")
for prefix, count in prefix_counts.items():
    print(f"  {prefix}: {count}")

mhc = mhc[mhc['allele'].str.startswith('HLA-')].reset_index(drop=True)

print(f"\nRemaining entries: {len(mhc):,}")

Non-human allele prefixes and their counts:
  BoLA: 18
  DLA: 3
  Eqca: 4
  FLA: 1
  Gaga: 8
  Gogo: 1
  H2: 14
  Mamu: 25
  Patr: 13
  Ptal: 1
  RT1: 1
  SLA: 20
  Trvu: 1

Remaining entries: 855,152


In [23]:
# Extract HLA alpha group from allele names
def extract_hla_alpha(allele):
    if pd.isnull(allele):
        return None
    m = re.match(r"HLA-([A-Z])", allele)
    if m:
        return m.group(1)
    return None

mhc['hla_alpha'] = mhc['allele'].apply(extract_hla_alpha)

# Count how many unique alleles per hla_alpha
alleles_per_alpha = mhc.groupby('hla_alpha')['allele'].nunique()
print("\nNumber of unique alleles per HLA alpha group:")
for alpha, count in alleles_per_alpha.items():
    print(f"  HLA-{alpha}: {count}")


Number of unique alleles per HLA alpha group:
  HLA-A: 68
  HLA-B: 108
  HLA-C: 32
  HLA-E: 2
  HLA-G: 3


## **2)Split into train and validation datasets**

In [24]:
print("\nHLA alpha distribution before splitting:")
print(mhc['hla_alpha'].value_counts())
print(f"Total: {len(mhc):,}\n")

# 85/15 stratified split on hla_alpha; IEDB will be used as the test set
train_class1, val_class1 = train_test_split(
    mhc,
    test_size=0.15,
    random_state=42,
    stratify=mhc['hla_alpha']
)

print("After stratified splitting (MHC train/validation):")
print(f"\nTrain: {len(train_class1):,} rows ({len(train_class1)/len(mhc)*100:.1f}%)")
print(train_class1['hla_alpha'].value_counts())
print("Proportions:", train_class1['hla_alpha'].value_counts(normalize=True).to_dict())

print(f"\nValidation: {len(val_class1):,} rows ({len(val_class1)/len(mhc)*100:.1f}%)")
print(val_class1['hla_alpha'].value_counts())
print("Proportions:", val_class1['hla_alpha'].value_counts(normalize=True).to_dict())



HLA alpha distribution before splitting:
hla_alpha
B    419578
A    333866
C     99140
G      2499
E        69
Name: count, dtype: int64
Total: 855,152

After stratified splitting (MHC train/validation):

Train: 726,879 rows (85.0%)
hla_alpha
B    356641
A    283786
C     84269
G      2124
E        59
Name: count, dtype: int64
Proportions: {'B': 0.490646999019094, 'A': 0.39041711206404367, 'C': 0.11593263803191453, 'G': 0.0029220819421114106, 'E': 8.116894283642808e-05}

Validation: 128,273 rows (15.0%)
hla_alpha
B    62937
A    50080
C    14871
G      375
E       10
Name: count, dtype: int64
Proportions: {'B': 0.49064885049854606, 'A': 0.39041731307445837, 'C': 0.115932425374007, 'G': 0.0029234523243394945, 'E': 7.795872864905319e-05}


In [ ]:
# Save the cleaned full MHC Class I CSV for downstream steps
os.makedirs("data", exist_ok=True)
mhc.to_csv("data/mhc_class1.csv", index=False)

print("Saved full Class I CSV: data/mhc_class1.csv")


Saved full Class I CSV: data/mhc_class1.csv


: 

## **2. Cleaning and Preparing IEDB Data**
     a) Remove non-human alleles
     b) Drop duplicates found in MHCflurry's IEDB export and remove NULL values
     c) Filter to Class 1 HLA alleles
     d) Use cleaned IEDB as the external test split

In [ ]:
# Load IEDB data from MHCflurry downloads (already downloaded!)
iedb_path = downloads.get_path("data_iedb", "mhc_ligand_full.csv.bz2")

iedb = pd.read_csv(iedb_path)

print(f"IEDB data loaded successfully!")
print(f"Shape: {iedb.shape[0]:,} rows × {iedb.shape[1]} columns")
print(f"\nColumns: {list(iedb.columns)}")
print(f"\nFirst 5 rows:")
iedb.head()

## **a) Remove non-human alleles**

In [ ]:
human_iedb = iedb[iedb['MHC Restriction'].str.startswith('HLA-', na=False)]
print(f"Shape: {human_iedb.shape[0]:,} rows × {human_iedb.shape[1]} columns")
human_iedb.head()

Shape: 2,599,797 rows × 111 columns


,Assay ID,Reference,Reference.1,Reference.2,Reference.3,Reference.4,Reference.5,Reference.6,Reference.7,Epitope,...,Assay.12,Antigen Presenting Cell,Antigen Presenting Cell.1,Antigen Presenting Cell.2,Antigen Presenting Cell.3,Antigen Presenting Cell.4,MHC Restriction,MHC Restriction.1,MHC Restriction.2,MHC Restriction.3
1,http://www.iedb.org/assay/26,http://www.iedb.org/reference/274,Literature,15448372,NaN,Yi-Hsiang Huang; Mi-Hua Tao; Cheng-Po Hu; Wan-...,J Gen Virol,2004,Identification of novel HLA-A*0201-restricted ...,http://www.iedb.org/epitope/31803,...,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*02:01,http://purl.obolibrary.org/obo/MRO_0001007,NaN,I
2,http://www.iedb.org/assay/115,http://www.iedb.org/reference/299,Literature,15140958,NaN,Yue-Dan Wang; Wan-Yee Fion Sin; Guo-Bing Xu; H...,J Virol,2004,T-cell epitopes in severe acute respiratory sy...,http://www.iedb.org/epitope/36724,...,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A2,http://purl.obolibrary.org/obo/MRO_0001530,NaN,I
5,http://www.iedb.org/assay/247,http://www.iedb.org/reference/329,Literature,15104671,NaN,C Sylvester-Hvid; M Nielsen; K Lamberth; G Rød...,Tissue Antigens,2004,"SARS CTL vaccine candidates; HLA supertype-, g...",http://www.iedb.org/epitope/14829,...,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*03:01,http://purl.obolibrary.org/obo/MRO_0001026,NaN,I
6,http://www.iedb.org/assay/248,http://www.iedb.org/reference/329,Literature,15104671,NaN,C Sylvester-Hvid; M Nielsen; K Lamberth; G Rød...,Tissue Antigens,2004,"SARS CTL vaccine candidates; HLA supertype-, g...",http://www.iedb.org/epitope/14829,...,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*11:01,http://purl.obolibrary.org/obo/MRO_0001029,NaN,I
7,http://www.iedb.org/assay/249,http://www.iedb.org/reference/329,Literature,15104671,NaN,C Sylvester-Hvid; M Nielsen; K Lamberth; G Rød...,Tissue Antigens,2004,"SARS CTL vaccine candidates; HLA supertype-, g...",http://www.iedb.org/epitope/33667,...,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*03:01,http://purl.obolibrary.org/obo/MRO_0001026,NaN,I


In [ ]:
# Reuse the existing helper to extract the HLA alpha group from MHC Restriction.
# Ensure `human_iedb` is a true copy (not a view) to avoid SettingWithCopyWarning
human_iedb = human_iedb.copy()
# Use .loc to assign a new column safely
human_iedb.loc[:, 'hla_alpha'] = human_iedb['MHC Restriction'].apply(extract_hla_alpha)


## **b) Drop duplicates and NULL values**

In [ ]:
duplicate_cols = ['MHC Restriction', 'Epitope.5']
human_iedb = human_iedb.dropna(subset=duplicate_cols).copy()

# Check duplicate allele/peptide pairs in the MHCflurry IEDB export.
duplicate_mask = human_iedb.duplicated(subset=duplicate_cols, keep='first')
duplicates_removed = int(duplicate_mask.sum())

print(f"Rows before deduplication: {len(human_iedb):,}")
print(f"Duplicate allele/peptide pairs found in the MHCflurry IEDB export: {duplicates_removed:,}")
print(f"Columns used for comparison: {duplicate_cols}")

human_iedb = human_iedb.loc[~duplicate_mask].reset_index(drop=True)
print(f"Rows after deduplication: {len(human_iedb):,}")

Rows before deduplication: 135,566
Duplicate allele/peptide pairs found in the MHCflurry IEDB export: 134,452
Columns used for comparison: ['MHC Restriction', 'Epitope.5']
Rows after deduplication: 1,114


## **c) Filter to Class 1 HLA alleles**

In [ ]:
iedb_class1 = human_iedb[human_iedb['MHC Restriction.3'] == "I"].copy()

alleles_per_alpha = iedb_class1.groupby('hla_alpha')['MHC Restriction'].count()
print("\nNumber of class I alleles (total count, not unique) per HLA alpha group:")
for alpha, count in alleles_per_alpha.items():
    print(f"  HLA-{alpha}: {count}")

print(f"\nClass I rows retained: {iedb_class1.shape[0]:,}")



Number of class I alleles (total count, not unique) per HLA alpha group:
  HLA-A: 245
  HLA-B: 334
  HLA-C: 123

Class I rows retained: 702


## **d) Use cleaned IEDB as the external test split**

In [ ]:
print("REMOVING OVERLAPS: IEDB Class I vs MHC Train/Validation Set")
print("="*80)

iedb_peptide_col = 'Epitope.5'  # Linear peptide sequence in IEDB
iedb_allele_col = 'MHC Restriction'  # HLA allele in IEDB
mhc_peptide_col = 'peptide'
mhc_allele_col = 'allele'
mhc_seen = pd.concat([train_class1, val_class1], ignore_index=True)

# Check if columns exist
print(f"\n1. Verifying columns...")
print(f"   IEDB columns available: {iedb_peptide_col in iedb_class1.columns and iedb_allele_col in iedb_class1.columns}")
print(f"   MHC columns available: {mhc_peptide_col in mhc_seen.columns and mhc_allele_col in mhc_seen.columns}")

# Function to normalize for comparison
def normalize_string(s):
    """Normalize string: strip whitespace, uppercase, handle nulls"""
    if pd.isna(s):
        return ""
    return str(s).strip().upper()

# Create MHC train/validation pairs (peptide, allele)
print(f"\n2. Creating MHC train/validation pairs...")
test_pairs = set()
for _, row in mhc_seen.iterrows():
    peptide = normalize_string(row[mhc_peptide_col])
    allele = normalize_string(row[mhc_allele_col])
    if peptide and allele:  # Skip empty pairs
        test_pairs.add((peptide, allele))

print(f"MHC train/validation pairs: {len(test_pairs):,}")

# Find overlaps in IEDB
print(f"\n3. Checking IEDB for overlaps...")
print(f"   IEDB Class I before: {len(iedb_class1):,} rows")

overlap_mask = []
for _, row in iedb_class1.iterrows():
    peptide = normalize_string(row[iedb_peptide_col])
    allele = normalize_string(row[iedb_allele_col])
    is_overlap = (peptide, allele) in test_pairs
    overlap_mask.append(is_overlap)

overlap_count = sum(overlap_mask)
print(f"   Overlaps found: {overlap_count:,} ({overlap_count/len(iedb_class1)*100:.2f}%)")

# Remove overlaps
iedb_class1_clean = iedb_class1[~pd.Series(overlap_mask, index=iedb_class1.index)].copy()

print(f"\n4. Results:")
print(f"   IEDB Class I after: {len(iedb_class1_clean):,} rows")
print(f"   Removed: {len(iedb_class1) - len(iedb_class1_clean):,} rows")
print(f"   Retention: {len(iedb_class1_clean)/len(iedb_class1)*100:.1f}%")

# Verify no overlaps remain
print(f"\n5. Verification...")
iedb_pairs_check = set()
for _, row in iedb_class1_clean.iterrows():
    peptide = normalize_string(row[iedb_peptide_col])
    allele = normalize_string(row[iedb_allele_col])
    if peptide and allele:
        iedb_pairs_check.add((peptide, allele))

remaining_overlaps = len(test_pairs & iedb_pairs_check)
print(f"   Remaining overlaps: {remaining_overlaps} (should be 0)")

if remaining_overlaps == 0:
    print(f"   SUCCESS: No overlaps between IEDB and MHC train/validation set")
else:
    print(f"   WARNING: {remaining_overlaps} overlaps still present!")

# Update the variable
iedb_class1 = iedb_class1_clean

REMOVING OVERLAPS: IEDB Class I vs MHC Train/Validation Set

1. Verifying columns...
   IEDB columns available: True
   MHC columns available: True

2. Creating MHC train/validation pairs...
   MHC train/validation pairs: 812,117

3. Checking IEDB for overlaps...
   IEDB Class I before: 702 rows
   Overlaps found: 0 (0.00%)

4. Results:
   IEDB Class I after: 702 rows
   Removed: 0 rows
   Retention: 100.0%

5. Verification...
   Remaining overlaps: 0 (should be 0)
   SUCCESS: No overlaps between IEDB and MHC train/validation set


## **Saving data into CSV**

In [ ]:
# Save unified MHC train/val splits and cleaned IEDB test split to data/
os.makedirs("data", exist_ok=True)
train_class1.to_csv("data/mhc1_train.csv", index=False)
val_class1.to_csv("data/mhc1_val.csv", index=False)
iedb_class1.to_csv("data/mhc1_test.csv", index=False)

# Save cleaned IEDB class I data to data/
iedb_class1.to_csv("data/iedb_class1.csv", index=False)

# Save canonical IEDB NPZ export for AR consumption
iedb_npz = {
    "allele": iedb_class1["MHC Restriction"].astype(str).to_numpy(),
    "peptide": iedb_class1["Epitope.5"].astype(str).to_numpy(),
    # The MHCflurry ligand export has no affinity targets, so use a positive placeholder.
    "measurement_value": np.ones(len(iedb_class1), dtype=np.float32),
    "split": np.full(len(iedb_class1), "test"),
}

np.savez_compressed("data/iedb.npz", **iedb_npz)

print("Saved: data/mhc1_train.csv, data/mhc1_val.csv, data/mhc1_test.csv, data/iedb_class1.csv, data/iedb.npz")

Saved: data/mhc1_train.csv, data/mhc1_val.csv, data/mhc1_test.csv, data/iedb_class1.csv, data/iedb.npz


## **AR Generator Training Dataset (Balanced MS NPZ)**

The balanced AR-training NPZ is prepared directly from this notebook.


In [ ]:
AR_CSV = Path("data/mhc_class1.csv")
AR_OUT = Path("data/mhc_class1_ms_balanced.npz")
AR_CAP = 2000
AR_MIN_BUCKET = 30
AR_VAL_FRAC = 0.10
AR_SEED = 42
STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

print("AR data prep config:")
print(f"  Input CSV: {AR_CSV}")
print(f"  Output NPZ: {AR_OUT}")
print(f"  cap={AR_CAP}, min_bucket={AR_MIN_BUCKET}, val_frac={AR_VAL_FRAC}, seed={AR_SEED}")


AR data prep config:
  Input CSV: data/mhc_class1.csv
  Output NPZ: data/mhc_class1_ms_balanced.npz
  cap=2000, min_bucket=30, val_frac=0.1, seed=42


In [ ]:
df_ar = pd.read_csv(AR_CSV).copy()
df_ar = df_ar.dropna(subset=["allele", "peptide", "measurement_value"])
df_ar = df_ar[df_ar["peptide"].apply(lambda p: all(c in STANDARD_AAS for c in str(p)))]
df_ar = df_ar.drop_duplicates(subset=["allele", "peptide"]).copy()

save_dict = {
    "allele": df_ar["allele"].astype(str).to_numpy(),
    "peptide": df_ar["peptide"].astype(str).to_numpy(),
    "measurement_value": df_ar["measurement_value"].astype(np.float32).to_numpy(),
}
np.savez_compressed(AR_OUT, **save_dict)